# PC-Learn Authentication Blueprint

> **Status:** Design / Blueprint  
> **Rule:** No implementation code yet. This notebook describes what the authentication system must do before coding begins.

---

## 1. Purpose

The authentication system is responsible for:

- Creating user accounts
- Activating accounts through email
- Logging users in
- Logging users out
- Supporting Google authentication
- Managing user roles
- Handling requests for protected roles such as Teacher and Author
- Controlling API access through permissions selected directly in API/View logic
- Providing JWT-based authentication for the React frontend

The system should be designed so that **roles describe what a user is**, while the actual API/View decides **what that role is allowed to do for that endpoint**.


# 2. Core Design Principles

## 2.1 Every registered user is a Member

There is no separate `Student` or `Learner` role.

A registered account automatically has:

**Member**

Additional roles can be added:

- Teacher
- Author

A user can therefore have multiple roles.

### Valid examples

| User | Roles |
|---|---|
| User A | Member |
| User B | Member + Teacher |
| User C | Member + Author |
| User D | Member + Teacher + Author |

`Member` cannot be removed.

Teacher and Author can be removed by the user.

---

## 2.2 Roles do not directly contain API permissions

A `Role` only contains:

- `id`
- `name`

The role does **not** contain a large permission matrix.

Instead, permissions are selected directly in the API/View.

Example concept:

```text
Teacher role
    ↓
API/View checks:
    "Does this user have Teacher?"
    ↓
Allow or deny
```

This gives direct control over each endpoint and keeps authorization logic close to the API behavior.

The exact permission classes/helpers will be designed during implementation.

---

## 2.3 Admin

Admin is conceptually the administrator of the entire platform.

The current design treats Admin as the **Django SuperUser / Django administrative authority**, rather than necessarily creating a duplicate custom `Admin` role.

Admin can:

- Manage users
- Accept or reject Teacher requests
- Accept or reject Author requests
- Modify courses
- Modify articles/blogs
- Control platform data
- Access administrative functionality
- Override normal ownership restrictions where appropriate

The exact Django permission strategy will be finalized during implementation.


# 3. Main Authentication Entities

The authentication system has four important conceptual entities:

1. `UserBase`
2. `Role`
3. `UserRole`
4. `RoleRequest`

The relationship is:

```text
UserBase
   │
   ├── UserRole ──── Role
   │
   └── RoleRequest ── Role
```

### Meaning

**UserBase**
> Who is this account?

**Role**
> What types of capabilities/positions exist?

**UserRole**
> Which roles does this user currently have?

**RoleRequest**
> Which protected role is this user asking an admin to grant?


# 4. UserBase

## Purpose

`UserBase` is the main custom user model.

It stores account and identity information.

It should **not** store separate Boolean fields such as:

```text
is_teacher
is_author
is_member
```

because the project uses a many-role system.

## Main data

Initial design:

| Field | Purpose |
|---|---|
| `id` | Unique user identifier |
| `UserName` | Login username, unique |
| `FName` | First name |
| `LName` | Last name |
| `Email` | User email |
| `is_staff` | Django staff status |
| `is_active` | Account activation/status |
| password | Django password handling |
| timestamps | Account-related dates if needed |

The exact fields can be adjusted before migrations are created.

## Authentication identity

Current design:

```text
USERNAME_FIELD = UserName
```

Therefore normal login uses:

**Username + Password**


# 5. Role

## Purpose

`Role` represents a role available in the platform.

A Role is intentionally simple.

### Fields

| Field | Type / idea | Purpose |
|---|---|---|
| `id` | Primary key | Unique role ID |
| `name` | String, unique | Role name |

Examples:

```text
Member
Teacher
Author
```

### Important rule

The Role model does **not** contain API permissions.

Permissions are decided by individual APIs/Views.

This means the role table stays simple while authorization remains explicit and controllable.


# 6. UserRole

## Purpose

`UserRole` connects users and roles.

It represents:

> This user currently has this role.

This creates a many-to-many relationship:

```text
UserBase  ←→  Role
          through
        UserRole
```

## Main fields

| Field | Purpose |
|---|---|
| `id` | Unique relationship ID |
| `user` | Related UserBase |
| `role` | Related Role |

## Important rule

A user must not receive the same role twice.

Conceptually:

```text
(user, role) = unique
```

Example:

```text
Ali → Member
Ali → Teacher
Ali → Author
```

but never:

```text
Ali → Teacher
Ali → Teacher
```

## Member rule

Every normal registered user receives:

```text
Member
```

This role cannot be removed.

Additional roles are optional.


# 7. RoleRequest

## Purpose

`RoleRequest` handles requests for protected roles.

Currently:

- Teacher
- Author

A user may request one of these roles during signup or later.

The request is separate from the user's account activation.

---

## Why it is separate

Account activation answers:

> Does this user own/verify this account?

Role approval answers:

> Has an administrator approved this user for an additional role?

These are two different processes.

Therefore:

```text
Account Activation
        ≠
Role Approval
```

---

## Main fields

Initial design:

| Field | Purpose |
|---|---|
| `id` | Request ID |
| `user` | User making the request |
| `requested_role` | Teacher or Author |
| `description` | Why they want the role / what they can provide |
| `attachment(s)` | Optional evidence/material |
| `status` | Pending / Accepted / Rejected |
| `created_at` | Request creation time |
| `reviewed_at` | Time admin reviewed it |
| reviewer/admin | Admin who reviewed it, if needed |
| admin response | Optional explanation/response |

Exact field names and attachment architecture will be decided before implementation.


# 8. Role Request Lifecycle

The basic lifecycle is:

```text
User
  ↓
Creates RoleRequest
  ↓
Pending
  ↓
Admin reviews
  ├── Accepted
  │      ↓
  │   UserRole is created
  │
  └── Rejected
         ↓
      Cooldown applies
```

## Accepted

If the request is accepted:

```text
RoleRequest.status = Accepted
```

and the requested role is assigned to the user.

Example:

```text
User: Ali
Current roles: Member

Request:
    Teacher

Admin:
    Accept

Result:
    Member + Teacher
```

## Rejected

If rejected:

```text
RoleRequest.status = Rejected
```

The role is not added.

The request remains part of the history.

---

# 9. Reapplication / Cooldown

A rejected user should not be able to immediately submit another request for the same protected role.

A cooldown period is planned.

Initial idea:

```text
1 request per week
```

The exact rule is **not finalized yet**.

Possible future decisions include:

- One request per role per week
- One total role request per week
- Cooldown measured from rejection time
- Different rules for Teacher and Author

This should be finalized before implementation.


# 10. Role Rules

## Member

Every registered user is a Member.

Member permissions are not represented as a separate list inside the Role model.

The API/View decides what Members can do.

---

## Teacher

Teacher is an additional role.

Requirements:

- User must already have an active/verified account
- User requests Teacher
- Admin approves the request
- Teacher role is assigned

Teacher can fully CRUD **their own courses**, subject to the API rules.

---

## Author

Author is an additional role.

Requirements:

- User must already have an active/verified account
- User requests Author
- Admin approves the request
- Author role is assigned

Author can fully CRUD **their own articles/blogs**, subject to the API rules.

---

## Teacher + Author

A user can have both roles.

Example:

```text
Member + Teacher + Author
```

Such a user can access APIs according to the permissions defined by each API/View.


# 11. Permission / Authorization Philosophy

This is a key architectural decision.

## Roles answer:

> "What role(s) does this user have?"

## API/View permission logic answers:

> "What may this user do here?"

Example:

```text
POST /courses/
```

could require:

```text
Teacher
```

while:

```text
GET /courses/
```

could allow:

```text
Member
Teacher
Author
```

Another endpoint might require:

```text
Teacher + ownership
```

For example:

```text
PATCH /courses/{id}/
```

could mean:

```text
User must be Teacher
AND
course.owner == request.user
```

This architecture allows each endpoint to define its own exact rules.

---

# 12. Generic Role Checking

The project may use a generic role-checking system rather than creating a separate permission class for every possible role combination.

Conceptual idea:

```text
IsRole("Teacher")
```

or equivalent logic.

It may later support:

### One required role

```text
Teacher
```

### Any of several roles

```text
Teacher OR Author
```

### Multiple required roles

```text
Teacher AND Member
```

Because every Teacher is also a Member, some combinations may be unnecessary.

The exact API will be designed during implementation.

---

# 13. Important Authorization Difference

Role checking and object ownership are different.

Example:

```text
Teacher
```

does not automatically mean:

> Can modify every course.

Instead:

```text
Teacher
+
owns this course
```

may be required.

Therefore authorization can contain multiple layers:

```text
Authentication
      ↓
Is the user logged in?
      ↓
Role check
      ↓
Does the user have the required role?
      ↓
Object/ownership check
      ↓
Can this user modify this specific object?
```

Admin can have broader authority according to the final admin permission design.


# 14. Normal Signup Flow

## Step 1: User submits signup data

The user provides the required signup information.

Example conceptual data:

```text
Username
First name
Last name
Email
Password
```

Optional role requests may also be included.

---

## Step 2: Account is created

The account is created as a Member.

Conceptually:

```text
UserBase created
        ↓
UserRole created:
    Member
```

---

## Step 3: Activation email

An activation email is sent.

The activation link follows this structure:

```text
http://{current_site}/account/activate/{uid}/{token}/
```

The token is based on Django's token-generation mechanism.

Conceptually:

```text
uid
+
activation token
```

---

## Step 4: User activates account

The user opens the activation link.

The account becomes active/verified according to the final activation implementation.

---

## Step 5: Login

The user can now log in normally using:

```text
Username
+
Password
```

---

## Step 6: Role request

If the user requested Teacher or Author:

```text
RoleRequest = Pending
```

The user waits for admin review.

Role approval does not need to happen before normal Member login.


# 15. Signup + Role Request

A user may request Teacher/Author during signup.

Example:

```text
Signup
  ↓
Create Member account
  ↓
Send activation email
  ↓
User activates account
  ↓
RoleRequest remains Pending
  ↓
Admin reviews request
```

Important:

**Requesting Teacher/Author does not automatically grant the role.**

The role only becomes active after admin approval.

This prevents users from granting themselves protected roles.


# 16. Login

## Normal login

Input:

```text
Username
Password
```

The backend authenticates the user.

On successful login, JWT tokens are returned.

The system uses:

- Access token
- Refresh token

The exact token lifetime/settings will be decided during implementation.

---

## Login requirements

At minimum, the system should consider:

- Correct username/password
- Account activation status
- Account active status
- Invalid credentials
- Token creation


# 17. Logout

The logout flow uses the refresh token.

Conceptually:

```text
Client
  ↓
Sends refresh token
  ↓
Backend blacklists refresh token
  ↓
Token can no longer be used
```

The project uses JWT token blacklisting.

A built-in JWT refresh endpoint is also planned.

The exact endpoint names will be finalized in the API design stage.


# 18. Google Authentication

Google authentication can be used for both:

- Signup
- Login

There is no separate email-verification step for Google authentication in the current design.

Reason:

The project treats Google's account/email verification as sufficient proof of control of the Google account.

---

## Google signup

Conceptually:

```text
Google authentication
        ↓
User does not exist
        ↓
Create UserBase
        ↓
Assign Member
        ↓
Login
```

## Google login

```text
Google authentication
        ↓
User already exists
        ↓
Authenticate
        ↓
Login
```

The exact Google OAuth/token-validation implementation will be designed later.


# 19. Email Verification vs Account Activation

The project currently uses the following terminology/process:

### Normal signup

```text
Signup
  ↓
Email activation link
  ↓
User activates account
  ↓
Account becomes active/verified
```

The activation URL is:

```text
/account/activate/{uid}/{token}/
```

Google authentication follows a separate path and does not require the normal activation email flow.


# 20. Authentication API Plan

Initial API/view list:

| Function | Purpose |
|---|---|
| Signup | Create a normal account |
| Login | Authenticate username/password |
| Logout | Blacklist refresh token |
| Token Refresh | Obtain a new access token |
| Google Auth | Google signup/login |
| Email Activation | Activate account through email |
| Role Request | Request Teacher/Author |
| Role Request List/Detail | View request status/history |
| Role Management | Add/remove allowed additional roles |
| User Information | Return authenticated user's information |

Exact URLs, HTTP methods, serializers, ViewSets, and response structures will be designed later.

---

# 21. Serializer Plan

Initial serializer concepts:

## UserBaseSerializer

Main serializer for normal user signup and/or user data.

Responsibilities will be finalized later.

Potential responsibilities:

- Validate signup input
- Create user
- Hash password correctly
- Assign Member
- Trigger activation flow
- Handle role request information if signup supports it

---

## UserTokenObtainSerializer

A serializer based on the project's JWT token-obtain serializer.

Purpose:

```text
Username + Password
        ↓
JWT authentication
```

The exact inheritance and customization will be decided during implementation.


# 22. View Architecture

The project will use **Django REST Framework Generic ViewSets** as the preferred API architecture.

The exact ViewSet structure is not finalized yet.

The important principle is:

```text
ViewSet
   ↓
Serializer
   ↓
Business logic / validation
   ↓
Permission checks
   ↓
Model/database
```

Authorization should be explicit and readable inside the API design.

This matches the project's preference for controlling permissions at the API/View level.


# 23. Authentication States

The system should distinguish these concepts:

### Account state

```text
Inactive / not activated
Active / activated
```

### Role request state

```text
Pending
Accepted
Rejected
```

### Current roles

```text
Member
Member + Teacher
Member + Author
Member + Teacher + Author
```

These states should not be mixed together.

For example:

```text
Active Member
+
Pending Teacher Request
```

is completely valid.

The user can still use Member functionality while waiting for Teacher approval.


# 24. Important Edge Cases

The implementation must eventually handle cases such as:

### Signup

- Duplicate username
- Duplicate/invalid email according to final email rules
- Invalid password
- Missing required data
- Password confirmation mismatch, if used

### Activation

- Invalid UID
- Invalid token
- Expired/invalid token
- Already activated account
- Nonexistent user

### Login

- Wrong username
- Wrong password
- Inactive account
- Unverified account, depending on final rule

### Google

- Google account already linked
- Google authentication failure
- Existing local account with the same email
- Invalid Google credential

### Roles

- User already has requested role
- User tries to request Member
- User tries to request Admin
- User submits duplicate pending request
- User submits request during cooldown
- User attempts to assign themselves Teacher/Author
- User attempts to remove Member
- User attempts to assign a role without admin approval

### Role requests

- Admin reviews an already-reviewed request
- Request belongs to another user
- Invalid requested role
- Missing required description
- Invalid attachment
- Accepted request where the role is already assigned


# 25. Security Rules

The authentication system should follow these principles:

1. Passwords are never stored directly.
2. Django's password hashing system is used.
3. Users cannot grant themselves protected roles.
4. Teacher/Author roles require admin approval.
5. Refresh tokens are blacklisted on logout.
6. Activation tokens must be validated.
7. Object ownership must be checked where required.
8. API permissions must be enforced server-side.
9. Frontend restrictions are not considered security.
10. Sensitive authentication information should not be exposed unnecessarily.

The React frontend is a client of the backend, not the authority for permissions.


# 26. React ↔ Django Communication

Frontend:

**React**

Backend:

**Django + Django REST Framework**

Communication:

**Axios**

Conceptual flow:

```text
React
  ↓
Axios
  ↓
Django REST API
  ↓
Authentication / Permission checks
  ↓
Database
```

The backend remains responsible for authentication and authorization.

The React application should display the permissions/capabilities returned by the backend, but it must never be trusted as the security layer.


# 27. Role Data Examples

## Example 1: Normal member

```text
User:
    Ali

Roles:
    Member

Role Requests:
    none
```

---

## Example 2: Member requesting Teacher

```text
User:
    Ali

Roles:
    Member

Role Request:
    Teacher
    Status: Pending
```

Ali is still only a Member until approval.

---

## Example 3: Teacher approved

```text
User:
    Ali

Roles:
    Member
    Teacher

Role Request:
    Teacher
    Status: Accepted
```

---

## Example 4: Teacher + Author

```text
User:
    Ali

Roles:
    Member
    Teacher
    Author
```

The APIs decide what Ali can do based on the roles required by each endpoint.


# 28. Conceptual Database Relationship

```text
┌──────────────────────┐
│       UserBase       │
├──────────────────────┤
│ id                   │
│ UserName             │
│ FName                │
│ LName                │
│ Email                │
│ password             │
│ is_active            │
│ is_staff             │
│ ...                  │
└──────────┬───────────┘
           │
           │ 1:N
           ▼
┌──────────────────────┐
│      UserRole        │
├──────────────────────┤
│ id                   │
│ user_id              │
│ role_id              │
└──────────┬───────────┘
           │
           │ N:1
           ▼
┌──────────────────────┐
│        Role          │
├──────────────────────┤
│ id                   │
│ name                 │
└──────────────────────┘


┌──────────────────────┐
│       UserBase       │
└──────────┬───────────┘
           │
           │ 1:N
           ▼
┌──────────────────────┐
│     RoleRequest      │
├──────────────────────┤
│ id                   │
│ user_id              │
│ requested_role_id    │
│ description          │
│ attachments          │
│ status               │
│ created_at           │
│ reviewed_at          │
│ reviewer             │
│ admin_response       │
└──────────────────────┘
```


# 29. Old Prototype: What We Are Reusing Conceptually

The old authentication prototype is treated as **reference material**, not as code to copy directly.

Useful ideas extracted from it:

- Custom `UserBase`
- Custom user manager
- Separate `Role`
- Separate `UserRole` relationship
- `PermissionsMixin`
- JWT authentication
- Django's authentication infrastructure

Ideas that need redesign for the new project:

- Role request system
- Teacher/Author approval
- Role cooldown
- API-level permission design
- Current account activation flow
- Google authentication flow
- Exact user fields
- Exact admin strategy
- Exact ViewSet structure

The new system should be designed from requirements rather than being a cleaned-up copy of the old prototype.


# 30. Final Authentication Architecture

At a high level:

```text
                    ┌───────────────┐
                    │    UserBase   │
                    └───────┬───────┘
                            │
                 ┌──────────┴──────────┐
                 │                     │
                 ▼                     ▼
          ┌─────────────┐       ┌─────────────┐
          │  UserRole   │       │ RoleRequest │
          └──────┬──────┘       └──────┬──────┘
                 │                     │
                 ▼                     ▼
          ┌─────────────┐       ┌─────────────┐
          │    Role     │       │    Role     │
          └─────────────┘       └─────────────┘


Authentication:

Signup
  ↓
Member
  ↓
Email Activation
  ↓
Login
  ↓
JWT


Optional protected role:

User
  ↓
Role Request
  ↓
Admin Review
  ├── Accept → UserRole
  └── Reject → Cooldown


Authorization:

Request
  ↓
Authentication
  ↓
ViewSet
  ↓
Role Permission
  ↓
Ownership / object permission
  ↓
Allow / Deny
```

---

# 31. Current Decisions

These decisions are considered part of the current blueprint:

- [x] Every registered user is a Member
- [x] No Student/Learner role
- [x] Users can have multiple roles
- [x] Member cannot be removed
- [x] Teacher requires admin approval
- [x] Author requires admin approval
- [x] Teacher and Author can coexist
- [x] Role contains only ID and name
- [x] Permissions are selected at API/View level
- [x] RoleRequest is separate from UserRole
- [x] Account activation is separate from role approval
- [x] Normal signup uses email activation
- [x] Google can handle both signup and login
- [x] Google does not use the normal activation email flow
- [x] Logout blacklists the refresh token
- [x] React is the frontend
- [x] Django/DRF is the backend
- [x] Axios connects frontend to backend
- [x] Generic ViewSets are preferred
- [x] Old authentication code is reference material, not a template to copy

---

# 32. Decisions Still Open

Before writing the actual authentication code, these points should be finalized:

1. Exact `UserBase` fields
2. Whether `Email` is required and whether it is unique
3. Exact account activation/verification field strategy
4. Exact JWT settings and token lifetimes
5. Exact Google authentication implementation
6. Exact RoleRequest fields
7. Attachment storage rules and allowed file types
8. Exact role-request cooldown rule
9. Whether a user can have multiple pending requests for different roles
10. Exact admin permission architecture
11. Exact generic role-checking API
12. Exact ownership permission pattern
13. Exact API URLs
14. Exact serializers
15. Exact ViewSet structure
16. Error response format
17. Email templates
18. Rate limiting / abuse protection
19. Password reset flow
20. Account deletion/deactivation behavior

These are **design questions**, not implementation problems yet.

The next stage should resolve them one by one before creating the Django project structure.
